In [119]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context='notebook', style='darkgrid', font='sans-serif')

In [120]:
df = pd.read_csv('../files/6_class_csv.csv')

In [121]:
df.shape

(240, 7)

In [122]:
df.head()

,Temperature (K),Luminosity(L/Lo),Radius(R/Ro),Absolute magnitude(Mv),Star type,Star color,Spectral Class
0,3068,0.002400,0.1700,16.12,0,Red,M
1,3042,0.000500,0.1542,16.60,0,Red,M
2,2600,0.000300,0.1020,18.70,0,Red,M
3,2800,0.000200,0.1600,16.65,0,Red,M
4,1939,0.000138,0.1030,20.06,0,Red,M


In [123]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Temperature (K)         240 non-null    int64  
 1   Luminosity(L/Lo)        240 non-null    float64
 2   Radius(R/Ro)            240 non-null    float64
 3   Absolute magnitude(Mv)  240 non-null    float64
 4   Star type               240 non-null    int64  
 5   Star color              240 non-null    str    
 6   Spectral Class          240 non-null    str    
dtypes: float64(3), int64(2), str(2)
memory usage: 13.3 KB


In [124]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

In [125]:
X = df[['Temperature (K)', 'Luminosity(L/Lo)', 'Radius(R/Ro)', 'Absolute magnitude(Mv)']]
y = df['Star type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2 ,random_state=42, stratify=y)


In [126]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

print(f"Accuracy -> {round(knn.score(X_test, y_test), 4)*100}%")

Accuracy -> 68.75%


In [127]:
from sklearn.metrics import classification_report
print(classification_report(y_test, knn.predict(X_test)))

              precision    recall  f1-score   support

           0       0.71      0.62      0.67         8
           1       0.67      0.75      0.71         8
           2       0.70      0.88      0.78         8
           3       0.71      0.62      0.67         8
           4       0.67      0.50      0.57         8
           5       0.67      0.75      0.71         8

    accuracy                           0.69        48
   macro avg       0.69      0.69      0.68        48
weighted avg       0.69      0.69      0.68        48



In [128]:
df.describe()

,Temperature (K),Luminosity(L/Lo),Radius(R/Ro),Absolute magnitude(Mv),Star type
count,240.000000,240.000000,240.000000,240.000000,240.000000
mean,10497.462500,107188.361635,237.157781,4.382396,2.500000
std,9552.425037,179432.244940,517.155763,10.532512,1.711394
min,1939.000000,0.000080,0.008400,-11.920000,0.000000
25%,3344.250000,0.000865,0.102750,-6.232500,1.000000
50%,5776.000000,0.070500,0.762500,8.313000,2.500000
75%,15055.500000,198050.000000,42.750000,13.697500,4.000000
max,40000.000000,849420.000000,1948.500000,20.060000,5.000000


In [129]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sc = StandardScaler() # z = (x - mean) / std

knn = KNeighborsClassifier(n_neighbors=5)

pipe = Pipeline([
	('sc', sc),
	('knn', knn)
])

pipe.fit(X_train, y_train)
print(f"Accuracy -> {round(pipe.score(X_test, y_test), 4)*100}%")

Accuracy -> 95.83%


In [130]:
from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler() # x_new = (x - x_min) / (x_max - x_min)

pipe = Pipeline([
	('sc', mms),
	('knn', knn)
])

pipe.fit(X_train, y_train)
print(f"Accuracy-> {round(pipe.score(X_test, y_test), 4)*100}%")

Accuracy-> 95.83%


In [131]:
from sklearn.preprocessing import RobustScaler

rs = RobustScaler() # (x - meadian) / IQR

pipe = Pipeline([
	('sc', rs),
	('knn', knn)
])

pipe.fit(X_train, y_train)
print(f"Accuracy-> {round(pipe.score(X_test, y_test), 4)*100}%")

Accuracy-> 97.92%


In [132]:
 # Temperature (K), Radius(R/Ro) ushbu 2 ta feature bilan topish 
X_new = df[['Temperature (K)', 'Radius(R/Ro)']]
y_new = df['Star type']

X_tr, X_ts, y_tr, y_ts = train_test_split(X_new, y_new, test_size=0.2 ,random_state=42, stratify=y)


In [133]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_tr, y_tr)

print(f"Accuracy -> {round(knn.score(X_ts, y_ts), 4)*100}%")

Accuracy -> 54.169999999999995%


In [134]:
sc = StandardScaler() # z = (x - mean) / std

pipe = Pipeline([
	('sc', sc),
	('knn', knn)
])

pipe.fit(X_tr, y_tr)
print(f"Accuracy -> {round(pipe.score(X_ts, y_ts), 4)*100}%")

Accuracy -> 68.75%


In [135]:
rs = RobustScaler() # (x - meadian) / IQR

pipe = Pipeline([
	('sc', rs),
	('knn', knn)
])

pipe.fit(X_tr, y_tr)
print(f"Accuracy-> {round(pipe.score(X_ts, y_ts), 4)*100}%")

# Robust Scaler Standart Scalerga nisbatan yaxshiligini ko'rishimiz mumkin chunki accuracy 68.75% dan 83.33% ga ko'tarildi

Accuracy-> 83.33%


In [136]:
y_pred = pipe.predict(X_ts)

wrong_idx = y_ts[y_ts != y_pred].index
print(f"Xato indekslar:  {wrong_idx.tolist()}")

print(f"Xatolik -> {round(len(wrong_idx) / len(y_pred) * 100, 2)}%")


Xato indekslar:  [153, 122, 156, 120, 100, 216, 151, 60]
Xatolik -> 16.67%


In [137]:
# ======================================= K ni qanday tanlash kerak =======================================

# k=1 -> overfifting
# k=5 .... 21 -> muvozanat
# k=192 -> underfifting


In [143]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
result = []

for k in range(1, 42, 2):
    p = Pipeline([
        ('sc', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    score = cross_val_score(p, X_train, y_train, cv=cv).mean()
    result.append((k, round(score, 4)))
    print(f"k = {k}, score = {score}")

best_k, best_score = max(result, key=lambda x: x[1])
print(f"Eng yaxshi K: {best_k}  \nCV Score: {best_score * 100:.2f}%")

k = 1, score = 0.9894736842105264
k = 3, score = 0.9789473684210526
k = 5, score = 0.9894736842105264
k = 7, score = 0.9842105263157894
k = 9, score = 0.9738191632928477
k = 11, score = 0.9685560053981106
k = 13, score = 0.9738191632928477
k = 15, score = 0.9634278002699055
k = 17, score = 0.942375168690958
k = 19, score = 0.9265856950067477
k = 21, score = 0.9267206477732792
k = 23, score = 0.916464237516869
k = 25, score = 0.8692307692307694
k = 27, score = 0.8641025641025643
k = 29, score = 0.8642375168690958
k = 31, score = 0.8485829959514171
k = 33, score = 0.8538461538461538
k = 35, score = 0.8488529014844804
k = 37, score = 0.8383265856950068
k = 39, score = 0.8226720647773279
k = 41, score = 0.8226720647773279
Eng yaxshi K: 1  
CV Score: 98.95%


In [146]:
# ======================================= TUNING =======================================

from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ("sc", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 15, 21],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}

gs = GridSearchCV(pipe, grid, cv=cv, n_jobs=-1)  
gs.fit(X_train, y_train)

print("Eng yaxshi parametrlar:", gs.best_params_)
print(f"CV Score (Train): {gs.best_score_ * 100:.2f}%")
print(f"Test Accuracy: {gs.score(X_test, y_test) * 100:.2f}%")


Eng yaxshi parametrlar: {'knn__metric': 'manhattan', 'knn__n_neighbors': 7, 'knn__weights': 'distance'}
CV Score (Train): 100.00%
Test Accuracy: 100.00%
